맨 아래에 결론

# 데이터 전처리

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 사용 함수

In [2]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [3]:
# 중복값 확인
def check_id_duplicates(df, col_name, df_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 값 중복 확인")
    print(f"{'='*80}")
    
    print("전체 행 수:", len(df))
    print(f"{col_name} 고유 개수:", df[col_name].nunique())
    print(f"중복 {col_name} 개수:", df[col_name].duplicated().sum())

In [4]:
# 컬럼 분포 확인
def check_category_summary(df, df_name, col_name):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df) * 100).round(2)
    
    display(summary_df.head(10))

# 테이블 전처리

In [5]:
# ============================================================
# 1. 원본 데이터 로드
# ============================================================
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("data/profile.csv", index_col=0)
df_transcript = pd.read_csv("data/transcript.csv", index_col=0)

## df_portfolio 전처리

In [6]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[portfolio]")
df_portfolio_copy = df_portfolio.copy()
check_basic_info(df_portfolio_copy, "portfolio")

[portfolio]

portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
id,str,10,100.0,0,0.0,10
reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3


[테이블 요약]


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [7]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_portfolio_clean = (
    df_portfolio_copy
    .rename(columns={
    'id': 'offer_id',
    'reward': 'offer_reward'
    })
)
display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [8]:
# ============================================================
# channels 문자열에서 채널 포함 여부를 더미 컬럼으로 생성
# ============================================================
channel_list = ['web', 'email', 'mobile', 'social']


for ch in channel_list:
    df_portfolio_clean[ch] = (
        df_portfolio_clean['channels']
        .str
        .contains(ch)
        .astype(int)
    )

display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0


## df_portfolio 전처리 최종 확인

In [9]:
check_basic_info(df_portfolio_clean, "portfolio_clean")
check_id_duplicates(df_portfolio_clean, 'offer_id', 'portfolio_clean')


portfolio_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
offer_id,str,10,100.0,0,0.0,10
offer_reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3
web,int64,10,100.0,0,0.0,2
mobile,int64,10,100.0,0,0.0,2
social,int64,10,100.0,0,0.0,2
email,int64,10,100.0,0,0.0,1


[테이블 요약]


,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0



portfolio_clean의 값 중복 확인
전체 행 수: 10
offer_id 고유 개수: 10
중복 offer_id 개수: 0


## df_profile 전처리

In [10]:
# ============================================================
# 확인 및 복제
# ============================================================
print("\n[profile]")
df_profile_copy = df_profile.copy()
check_basic_info(df_profile_copy, "profile")


[profile]

profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,17000
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
income,float64,14825,87.21,2175,12.79,91
gender,str,14825,87.21,2175,12.79,3
id,str,17000,100.00,0,0.00,17000
became_member_on,int64,17000,100.00,0,0.00,1716
age,int64,17000,100.00,0,0.00,85


[테이블 요약]


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [11]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_profile_copy = (
    df_profile_copy
    .rename(columns={
    'id': 'customer_id'
}))

In [12]:
# ============================================================
# 가입일 날짜형 변환
# ============================================================
df_profile_copy['became_member_on'] = pd.to_datetime(
    df_profile_copy['became_member_on']
    .astype(str),
    format='%Y%m%d'
)

In [13]:
# ============================================================
# 가입일 파생 컬럼
# ============================================================
# 파생컬럼 필요시 사용
# # 가입 년도
# df_profile_copy['member_year'] = (
#     df_profile_copy['became_member_on']
#     .dt
#     .year
# )

# # 가입 월
# df_profile_copy['member_month'] = (
#     df_profile_copy['became_member_on'].
#     dt
#     .month
# )

# # 가입 분기
# df_profile_clean['member_quarter'] = (
#     df_profile_clean['became_member_on']
#     .dt
#     .quarter
# )

In [14]:
# ============================================================
# age 이상치 처리
# ============================================================
# age=118은 비정상값으로 보고 제거
# gender, income 결측도 함께 제거 대상에 포함


df_profile_copy['age'] = (
    df_profile_copy['age'].
    replace(118, np.nan)
)

# 제거 전 행 수 저장
before_len = len(df_profile_copy)

# age, gender, income 중 하나라도 결측이면 제거
df_profile_copy['age'] = df_profile_copy['age'].replace(118, np.nan)
df_profile_copy = df_profile_copy.dropna(subset=['age', 'gender', 'income']).copy()

# 제거 후 행 수 저장
after_len = len(df_profile_copy)

print("제거 전 행 수:", before_len)
print("제거 후 행 수:", after_len)
print("제거된 행 수:", before_len - after_len)

제거 전 행 수: 17000
제거 후 행 수: 14825
제거된 행 수: 2175


In [15]:
# ============================================================
# 컬럼 순서 정리
# ============================================================
df_profile_clean = df_profile_copy[
    [
        'customer_id',
        'gender', 'age', 'income', 'became_member_on'
    ]
]

## df_profile 전처리 최종확인

In [16]:
check_basic_info(df_profile_clean, "profile_clean")
check_id_duplicates(df_profile_clean, 'customer_id', 'profile_clean')


profile_clean의 컬럼 정보 / 결측치 확인 정보 요약



[전체 요약]


,항목,값
0,행 개수,14825
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
customer_id,str,14825,100.0,0,0.0,14825
became_member_on,datetime64[us],14825,100.0,0,0.0,1707
income,float64,14825,100.0,0,0.0,91
age,float64,14825,100.0,0,0.0,84
gender,str,14825,100.0,0,0.0,3


[테이블 요약]


,customer_id,gender,age,income,became_member_on
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,2017-07-15
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,2017-05-09
5,e2127556f4f64592b11af22de27a7932,M,68.0,70000.0,2018-04-26
8,389bc3fa690240e798340f5a15918d5c,M,65.0,53000.0,2018-02-09
12,2eeac8d8feae4a8cad5a6af0499a211d,M,58.0,51000.0,2017-11-11



profile_clean의 값 중복 확인
전체 행 수: 14825
customer_id 고유 개수: 14825
중복 customer_id 개수: 0


## df_transcript 전처리

In [17]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[df_transcript]")
df_transcript_copy = df_transcript.copy()
check_basic_info(df_transcript_copy, "transcript")
check_category_summary(df_transcript_copy, "transcript", "event")

[df_transcript]

transcript의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,4
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
person,str,306534,100.0,0,0.0,17000
value,str,306534,100.0,0,0.0,5121
time,int64,306534,100.0,0,0.0,120
event,str,306534,100.0,0,0.0,4


[테이블 요약]


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0



transcript의 event 범주 확인


,event,개수,비율(%)
0,transaction,138953,45.33
1,offer received,76277,24.88
2,offer viewed,57725,18.83
3,offer completed,33579,10.95


In [18]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'person': 'customer_id'
}))

In [19]:
# ============================================================
# value 컬럼 파싱
# ============================================================
import ast
# ast란?

# ============================================================
# value 컬럼 딕셔너리로 형변환
# value 컬럼은 문자열처럼 보이지만 사실 딕셔너리 형태
# 먼저 컬럼을 딕셔너리의 형태로 변환한다.
# ============================================================
df_transcript_copy['value'] = (
    df_transcript_copy['value']
    .apply(ast.literal_eval))

print(df_transcript_copy['value'].apply(type).value_counts())

value
<class 'dict'>    306534
Name: count, dtype: int64


In [20]:
# ============================================================
# value 컬럼 키 구조 확인
# ============================================================
event_keys = sorted({
    key
    for d in df_transcript_copy['value']
    for key in d.keys()
})

print("value 컬럼 전체 key 종류:")
print(event_keys)

value 컬럼 전체 key 종류:
['amount', 'offer id', 'offer_id', 'reward']


In [21]:
# ============================================================
# value 컬럼(dict) 분리 및 주요 파생 컬럼 생성
# event별로 value 안에 들어 있는 값을 별도 컬럼으로 분리
# 'offer id'와 'offer_id'는 하나의 offer_id 컬럼으로 통합
# ============================================================
value_df = pd.DataFrame(df_transcript_copy['value'].tolist())
df_transcript_copy = pd.concat([df_transcript_copy, value_df], axis=1)

# offer id / offer_id 컬럼명을 하나로 통합
df_transcript_copy['offer_id'] = df_transcript_copy['offer id'].fillna(df_transcript_copy['offer_id'])

df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'reward': 'event_reward'
    })
)

# 주요 컬럼 확인
display(
    df_transcript_copy[
        [
            'customer_id', 
            'event', 
            'time', 
            'offer_id', 
            'amount', 
            'event_reward']]
        .head()
)

,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


In [22]:
# ============================================================
# event별 컬럼 결측 개수 확인
# ============================================================
print("="*80)
print("이벤트별 결측 확인")
print("="*80)

print(df_transcript_copy[['event', 'offer_id', 'amount', 'event_reward']].isna().groupby(df_transcript_copy['event']).sum())

이벤트별 결측 확인
                 event  offer_id  amount  event_reward
event                                                 
offer completed      0         0   33579             0
offer received       0         0   76277         76277
offer viewed         0         0   57725         57725
transaction          0    138953       0        138953


In [23]:
# ============================================================
# 중복 컬럼 결측 개수 확인
# ============================================================

# 중복 확인용 value 비교 컬럼 생성
df_transcript_copy['value_str'] = df_transcript_copy['value'].astype(str)

# 중복 기준
dup_cols = ['customer_id', 'event', 'time', 'offer_id', 'event_reward', 'value_str']

# 중복 행 추출
dup_df = df_transcript_copy[
    df_transcript_copy.duplicated(subset=dup_cols, keep=False)
].sort_values(['customer_id', 'time', 'offer_id'])

print("중복에 포함된 전체 행 수:", len(dup_df))
print()

print("[중복 event 분포]")
print(dup_df['event'].value_counts())
print()

print("[중복 횟수 분포]")
print(
    df_transcript_copy
    .groupby(dup_cols)
    .size()
    .reset_index(name='dup_count')
    .query("dup_count > 1")['dup_count']
    .value_counts()
    .sort_index()
)

display(
    dup_df[
        ['customer_id', 'event', 'time', 'offer_id', 'event_reward', 'value']
    ].head(5)
)

중복에 포함된 전체 행 수: 793

[중복 event 분포]
event
offer completed    793
Name: count, dtype: int64

[중복 횟수 분포]
dup_count
2    395
3      1
Name: count, dtype: int64


,customer_id,event,time,offer_id,event_reward,value
218058,00d7c95f793a4212af44e632fdc1e431,offer completed,504,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...
218060,00d7c95f793a4212af44e632fdc1e431,offer completed,504,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...
220133,01925607d99c460996c281f17cdbb9e2,offer completed,510,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
220134,01925607d99c460996c281f17cdbb9e2,offer completed,510,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
171646,01956670cf414b309675aa73368b94a9,offer completed,420,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...


offer completed 이벤트에서만 중복 문제 발생

중복구조: 
- 395개 경우는 2번씩 중복: 395 * 2 = 790
- 1개 경우는 3번 중복: 1 * 3 = 3
- 790 + 3 = 793

이는 향후 영향을 줄수 있기 때문에 중복 제거

In [24]:
# ============================================================
# 중복 제거
# ============================================================
df_transcript_drop = (
    df_transcript_copy
    .drop_duplicates(subset=dup_cols)
    .drop(columns=['offer id', 'value_str'], errors='ignore')
    .copy()
)

print("중복 제거 전 행 수:", len(df_transcript_copy))
print("중복 제거 후 행 수:", len(df_transcript_drop))
print("제거된 행 수:", len(df_transcript_copy) - len(df_transcript_drop))

중복 제거 전 행 수: 306534
중복 제거 후 행 수: 306137
제거된 행 수: 397


실제로 제거되는 행 수는 397개\
중복 제거할 때는 각 그룹에서 1개만 남기고 나머지를 지우기 때문

In [25]:
# ============================================================
# value 컬럼 드랍
# value 컬럼을 분리 및 주요 파생 컬럼 생성했기 때문에 제거
# ============================================================
df_transcript_drop = (
    df_transcript_drop
    .drop(columns=['value'], errors='ignore')
    .copy()
)

df_transcript_drop.columns

Index(['customer_id', 'event', 'time', 'amount', 'offer_id', 'event_reward'], dtype='str')

In [26]:
df_transcript_clean = df_transcript_drop[
    [
        'customer_id',
        'event',
        'time',
        'offer_id',
        'amount',
        'event_reward'
    ]
]

## df_transcript 전처리 최종확인

In [27]:
check_basic_info(df_transcript_clean, "transcript_clean")
check_id_duplicates(df_transcript_clean, 'customer_id', 'transcript_clean(customer_id)')
check_id_duplicates(df_transcript_clean, 'offer_id', 'transcript_clean(offer_id)')


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN



transcript_clean(customer_id)의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137

transcript_clean(offer_id)의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


# 구매효과 분석

In [29]:
check_basic_info(df_profile_clean, "transcript_clean")


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,14825
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
customer_id,str,14825,100.0,0,0.0,14825
became_member_on,datetime64[us],14825,100.0,0,0.0,1707
income,float64,14825,100.0,0,0.0,91
age,float64,14825,100.0,0,0.0,84
gender,str,14825,100.0,0,0.0,3


[테이블 요약]


,customer_id,gender,age,income,became_member_on
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,2017-07-15
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,2017-05-09
5,e2127556f4f64592b11af22de27a7932,M,68.0,70000.0,2018-04-26
8,389bc3fa690240e798340f5a15918d5c,M,65.0,53000.0,2018-02-09
12,2eeac8d8feae4a8cad5a6af0499a211d,M,58.0,51000.0,2017-11-11


In [30]:
check_basic_info(df_portfolio_clean, "transcript_clean")


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
offer_id,str,10,100.0,0,0.0,10
offer_reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3
web,int64,10,100.0,0,0.0,2
mobile,int64,10,100.0,0,0.0,2
social,int64,10,100.0,0,0.0,2
email,int64,10,100.0,0,0.0,1


[테이블 요약]


,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0


In [31]:
check_basic_info(df_transcript_clean, "transcript_clean")


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


## 의문 1. transaction이 특정 offer와 직접 연결되는가
> transaction 행에 offer_id가 들어있는가?

In [32]:
event_null = (
    df_transcript_clean[['event', 'offer_id', 'amount', 'event_reward']]
    .isna()
    .groupby(df_transcript_clean['event'])
    .sum()
)

print(df_transcript_clean['event'].value_counts())
display(event_null)

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64


,event,offer_id,amount,event_reward
event,,,,
offer completed,0,0,33182,0
offer received,0,0,76277,76277
offer viewed,0,0,57725,57725
transaction,0,138953,0,138953


각 컬럼 한줄 해설
- event : 고객에게 어떤 행동/이벤트가 발생했는지를 나타내는 컬럼
    - offer received : 고객이 프로모션 오퍼를 받았다는 의미
    - offer viewed : 고객이 받은 프로모션 오퍼를 실제로 열어봤다는 의미
    - offer completed : 고객이 프로모션 조건을 충족해 완료했다는 의미
    - transaction : 고객의 실제 결제가 발생했다는 의미
- offer_id : 어떤 프로모션 오퍼인지 구분하는 식별값
- amount : transaction 발생 시 기록되는 구매 금액
- event_reward : 프로모션 완료로 지급된 보상값 정보

### 의문1 해석 결론

transaction = 실제 구매 이벤트\
offer received / viewed / completed = 오퍼 관련 이벤트

transaction 행에 amount는 존재하나 offer_id가 전부 비워져있다.\
거래가 발생했다는 사실은 확인할 수 있어도 그 거래가 어떤 오퍼 때문인지는 직접 연결되어 있지 않다.

즉, 구매 정보와 오퍼 반응 정보는 같은 종류의 이벤트가 아니라 별도 이벤트로 관리되고 있으며,\
**특정 transaction이 특정 offer의 결과라고 바로 해석하기는 어렵다**.

`구매 발생`과 `프로모션 반응 발생`은 확인 가능하지만\
`이 구매는 A오퍼 때문`, `이 구매는 B오퍼 효과`같이 거래와 오퍼를 1:1로 직접 연결하는 데에는 한계가 있다.

결론:\
프로모션의 반응 패턴 분석에는 활용할 수 있으나, 특정 프로모션이 실제 구매를 유도했다 증명함엔 한계가 있다.

기존주제: `기존 주재엔 어떤 고객에게 어떤 프로모션을 주면 실제 구매로 이어지는가`\
-> 프로모션 또는 오퍼가 실제 구매를 만들었다고 인과적으로 말하기는 어렵다.

## 의문2 프로모션을 안 받은 고객을 확인할수 있는가?

In [33]:
total_customers = df_transcript_clean['customer_id'].nunique()

offer_received_customers = (
    df_transcript_clean.loc[
        df_transcript_clean['event'] == 'offer received',
        'customer_id'
    ]
    .nunique()
)

no_offer_customers = total_customers - offer_received_customers

print("전체 고객 수:", total_customers)
print("offer received 고객 수:", offer_received_customers)
print("offer 미수신 고객 수:", no_offer_customers)
print("offer 수신 비율(%):", round(offer_received_customers / total_customers * 100, 2))

전체 고객 수: 17000
offer received 고객 수: 16994
offer 미수신 고객 수: 6
offer 수신 비율(%): 99.96


### 의문2 해석 결론
프로모션을 아예 받지 않은 고객 수는 6명, 전체 고객 기준 오퍼 수신 비율은 99.96%

거의 모든 고객이 이미 프로모션에 노출되어 있었으며,\
프로모션을 받은 고객과 받지 않은 고객의 구매 행동을 비교하거나\
원래 구매 의사가 있었기 때문인지, 아니면 프로모션의 영향을 받았기 때문인지를 분리해서 해석하는 데 한계가 있다.

결론: \
어떤 고객에게 어떤 프로모션을 주면 실제 구매로 이어지는가\
는 프로모션을 받은 집단과 받지 않은 집단 간의 비교가 중요하나,
현재 데이터에서는 프로모션 미수신 고객이 거의 없어 이러한 비교 자체가 어렵다.

## 의문 3 offer viewed 없는 completed가 몇건인가

In [ ]:
# 1. offer별 최초 viewed 시간
first_view = (
    df_transcript_clean[df_transcript_clean['event'] == 'offer viewed']
    .groupby(['customer_id', 'offer_id'])['time']
    .min()
    .reset_index(name='first_view_time')
)

# 2. completed 데이터
completed = (
    df_transcript_clean[df_transcript_clean['event'] == 'offer completed']
    [['customer_id', 'offer_id', 'time']]
    .rename(columns={'time': 'comp_time'})
)

# 3. completed에 최초 viewed 시간 붙이기
result = completed.merge(
    first_view,
    on=['customer_id', 'offer_id'],
    how='left'
)

# 4. viewed 없는 completed 판별
no_prior_view = result[
    result['first_view_time'].isna() |
    (result['first_view_time'] > result['comp_time'])
]

print("offer completed 수:", len(completed))
print("viewed 없는 completed 수:", len(no_prior_view))
print("비율(%):", round(len(no_prior_view) / len(completed) * 100, 2))

offer completed 수: 33182
사전 viewed 없는 completed 수: 8568
비율(%): 25.82


### 의문 3 해석 결론
전체 offer completed 중 일부는 사전 offer viewed 기록 없이 발생했다.\
offer completed 33,182건 중 사전 offer viewed가 확인되지 않은 건수는 8,568건으로, 약 25.82%를 차지했다.\

즉, 일부 고객은 오퍼를 본 기록 없이도 완료 이벤트가 발생한 것으로,\
offer received → offer viewed → offer completed의 흐름이 항상 순차적으로 이어지지 않음을 보여준다.

따라서 이 데이터에서 viewed를 기준으로 반응 퍼널을 해석할 때\
단순 viewed를 거친 고객만이 completed에 도달한다고 가정하기 어려우며, 실제 반응 구조가 왜곡된다.

결론:\
프로모션 반응 흐름을 대략적으로 파악하는 데에는 활용할 수 있으나,\
반응 과정이 항상 received → viewed → completed의 일관된 흐름으로 이어진다고 보기는 어렵다.


## 의문 4 ROI나 순이익 계산이 가능한가?

In [ ]:
print("portfolio 컬럼:", df_portfolio_clean.columns.tolist())
print("transcript 컬럼:", df_transcript_clean.columns.tolist())

### 의문 4 해석 결론

계산 가능한 금액 컬럼

거래금액(transcript: amount, offer_reward, event_reward)

필요한 데이터가 없기 때문에 정확한 ROI, 순이익, 실제 수익성은 계산하기 어렵다.\

## 의문 5 거래 시점의 활성 오퍼 수
기존 주제에서 보고자 했던것:\
고객이 오퍼를 받았고 -> 그 뒤에 구매했고 -> 그 오퍼가 구매를 만들었다

하지만 transaction 행에는 offer_id가 없서 그 거래가 어떤 오퍼 때문인지 직접 알 수 없다.

만약...\ 
각 거래가 발생한 시점에서 그 고객에게 아직 유효한 오퍼가 몇 개 살아 있었는지 확인한다면\ 
**거래 시점에 유효한 오퍼가 1개만 있으면 그 오퍼 때문이라고 볼 수 있지 않나?**


### 의문 5 해석 결론

1. 0개인 거래가 있으면 -> 거래 시점에 유효한 오퍼가 하나도 없었다.\
오퍼 없이도 거래가 발생했다.\
모든 거래를 프로모션 효과로 볼 수는 없다

2. 1개인 거래가 있다면\
그 오퍼 때문에 구매했다는 결론을 내릴순 없다\
원래 살 사람이었을 수도 있고, 오퍼를 보고 산 것일 수도 있고, 그냥 우연히 시점만 겹쳤을 수도 있기 때문\
후보가 1개일 뿐이지 원인이 확정된 것은 아님

3. 2개 이상인 거래가 있으면 -> 거래 시점에 유효한 오퍼가 여러 개 동시에 있었다\
어떤 오퍼 때문인지, 어느 오퍼 영향이 더 컸는지 를 구분하기 어렵다.

# **결론**

지금의 데이터로 **고객이 프로모션에 어떻게 반응했는가 는** 볼 수 있지만,\
**이 프로모션이 실제 구매를 만들었다** 는 식으로 말하기에 근거가 부족하다.

반응 패턴 분석은 가능!\ 
다만, 구매 유도 효과 증명은 어려움

때문에 기존 주제인 **어떤 고객에게 어떤 프로모션을 주면 실제 구매로 이어지는가** 외의\
다른 주제를 선택하는것이 좋을것 같다.

의문 1~5 해석 결론을 종합하면,

**프로모션이 구매를 만들었다** 는 걸 어느 정도 설명해야 하지만, 지금 데이터론 그걸 말하기 어렵다.

예를들어\
예를 들어 어떤 고객이 월요일에 오퍼를 받았고, 수요일에 거래를 했다고 하자\
단순 "오퍼 받고 나서 샀네?"라 가정할순 있으나.

1. 원래 살 예정이었는데 그냥 산 것
2. 오퍼를 보고 산 것
3. 다른 오퍼의 영향을 받은 것
4. 오퍼와 상관없이 평소처럼 산 것

일 가능성이 있다. 현재의 데이터론 이 중 어느 경우인지 구분할수 없다.\
때문에 "오퍼 때문에 구매했다"라 주장할 명백한 근거가 없다.

지금 데이터론 보기 어려운 요소
1. 특정 오퍼가 특정 구매를 만들었다는 증명
2. 프로모션 효과의 인과 해석
3. 정확한 ROI, 순이익 계산

## 그렇다면 지금 데이터로 가능한 분석은??

## 고객별 프로모션 반응 분석
어떤 고객이 프로모션을 더 잘 보고, 완료하고, 거래까지 이어지는지 본다.

**어떤 고객이 프로모션에 더 잘 반응하는가?**

- 봐야할것?: 고객의 기본 속성에 따라 반응 차이
    - 성별 gender
    - 연령대 age
    - 소득 income
    - 가입 시점 became_member_on

- 확인해야할 지표?
    - 오퍼 수신 수
    - 오퍼 조회 수
    - 오퍼 완료 수
    - transaction 발생 수
    - view 전환율 = viewed / received
    - completion 전환율 = completed / received 또는 completed / viewed
    - transaction 반응률 = transaction 발생 고객 수 / received 고객 수

- 예시 해설
    - 30대 고객은 viewed 비율이 높다
    - 고소득 고객은 completed 비율이 높다
    - 가입 기간이 오래된 고객이 transaction 반응이 높다

> 어떤 고객 특성이 프로모션에 더 잘 반응하는가를 보기 좋음

## 오퍼 유형별 반응 차이 분석
어떤 오퍼 종류와 조건이 더 높은 반응을 만드는지 본다

**어떤 오퍼 특성이 더 높은 반응을 만드는가?**

- 봐야할것?: portfolio 기준으로 오퍼 특성
    - offer_type 별 차이
        - bogo
        - discount
        - informational
    - reward
    - difficulty
    - duration
    - channel

- 확인해야할 지표?
    - received 수
    - viewed 수
    - completed 수
    - transaction 발생 수


## 채널별/세그먼트별 반응 패턴 분석
어떤 고객군이 어떤 채널과 오퍼 조합에 더 잘 반응하는지 본다.

**어떤 고객군이 어떤 채널에서 더 잘 반응하는가?**

- 봐야할것?:채널, 세그먼트

- 지표?
    - viewed 비율
    - completed 비율
    - transaction 비율


- 해석 예시
    - 젊은 층은 mobile 중심 오퍼 반응이 높다
    - 중장년층은 email 반응이 상대적으로 높다

> 누구에게 어떤 방식으로 전달하는 게 더 반응이 좋은가? 를 보는 데 적합

##  viewed, completed, transaction의 분포와 흐름 파악
프로모션 반응이 실제로 어떤 순서와 구조로 나타나는지 본다

**프로모션 반응이 실제로 어떤 흐름으로 나타나는가?**

1. 퍼널 전환
    - received → viewed
    - viewed → completed
    - received → completed
    - completed 이후 transaction이 얼마나 동반되는지

2. 발견 가능한 비정상 흐름
    - viewed 없이 completed된 경우

3. 흐름 예시
    - received 후 viewed까지 걸린 시간
    - viewed 후 completed까지 걸린 시간
    - received 후 transaction까지 걸린 시간

- 봐야할 지표?
    - 이밴트 건수
    - 전환율
    - 평균 소요 시간
    - 이탈 구간

- 해석 예시
    - 많은 고객이 received 이후 viewed에서 이탈한다
    - viewed 이후 completed까지는 상대적으로 전환이 높다